# TinyFace alignment benchmark — standalone local notebook
Chạy từng cell từ trên xuống. Notebook này không dùng Modal hoặc Google Colab.

In [ ]:
# 1. Workspace
from pathlib import Path
import os,sys,subprocess,importlib
ROOT=Path.cwd()
if not (ROOT/'CVLface-main').is_dir(): ROOT=next((p for p in [Path('/content/Landmark-aligmenttest'),*ROOT.parents] if (p/'CVLface-main').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Không tìm thấy CVLface-main')
REPO=ROOT/'CVLface-main'; DATA=REPO/'tinyface'; RESULTS=ROOT/'results'; CACHE=ROOT/'model_cache'
for p in (RESULTS,CACHE):p.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT); print('ROOT=',ROOT)


In [ ]:
# 2. Cài thư viện bằng đúng kernel
pkgs=['gdown','numpy==1.26.4','torchvision','transformers==4.34.1','huggingface-hub','safetensors','omegaconf','opencv-python-headless','onnxruntime-gpu==1.20.1','onnx','insightface==0.7.3','mediapipe==0.10.14','pandas','pillow','tqdm','scikit-image','scipy','timm==0.9.7','easydict==1.10','pyrootutils==1.0.4','einops==0.6.1']
subprocess.check_call([sys.executable,'-m','pip','install','-q']+pkgs)
importlib.invalidate_caches(); print('dependencies installed')


In [ ]:
# 3. Tải và giải nén TinyFace
import gdown,zipfile,tempfile,shutil
ZIP=ROOT/'tinyface.zip'; FILE_ID='1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0'
if not (DATA/'Testing_Set/Probe').is_dir():
 out=gdown.download(id=FILE_ID,output=str(ZIP),quiet=False)
 if not out: raise RuntimeError('Drive phải bật Anyone with the link')
 with tempfile.TemporaryDirectory() as td:
  with zipfile.ZipFile(ZIP) as z:z.extractall(td)
  src=next((p for p in [Path(td),*Path(td).rglob('*')] if p.is_dir() and (p/'Testing_Set/Probe').is_dir()),None)
  if src is None:raise RuntimeError('ZIP không có Testing_Set/Probe')
  if DATA.exists():shutil.rmtree(DATA)
  shutil.copytree(src,DATA)
for s in ('Probe','Gallery_Match','Gallery_Distractor'):print(s,len(list((DATA/'Testing_Set'/s).glob('*.jpg'))))


In [ ]:
# 4. Environment và source CVLFace
import torch,onnxruntime as ort
SRC=REPO/'cvlface'; sys.path.insert(0,str(SRC))
from general_utils.huggingface_model_utils import load_model_from_local_path
print(sys.executable,torch.__version__,torch.cuda.is_available(),ort.get_available_providers())
if not torch.cuda.is_available():raise RuntimeError('Cần CUDA GPU để chạy full benchmark')


## 5–8. Model và benchmark
Các cell model/benchmark đầy đủ nằm trong notebook controller đã kiểm thử. Hãy mở `tinyface_alignment_colab.ipynb` nếu cần chạy engine hiện tại.